In [1]:
# Cell 1: 필수 라이브러리 및 Playwright 브라우저 설치
# 실행 환경에 Playwright와 pandas가 설치되어 있지 않다면 아래 셀을 먼저 실행하세요.
%pip install playwright pandas
!playwright install chromium


In [2]:
# 키워드 기반 상품 검색 크롤링
# 검색어를 입력받아 싸다구몰에서 최대 30개의 상품 정보를 수집합니다.
import sys
import tempfile
import subprocess
import os
import json
import re
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# 검색할 키워드 입력 (여기를 수정하세요)
SEARCH_KEYWORD = "스마트폰"  # 예시: "여성 티셔츠", "노트북", "스마트폰" 등

# 최대 수집할 상품 개수
MAX_PRODUCTS = 30

# 검색 URL 생성
search_url = f"https://ssadagu.kr/shop/search.php?ss_tx={quote(SEARCH_KEYWORD)}"

print(f"🔍 검색 키워드: {SEARCH_KEYWORD}")
print(f"🔗 검색 URL: {search_url}")
print(f"📦 최대 수집 개수: {MAX_PRODUCTS}개\n")

# 크롤링 스크립트 생성
crawl_script = f"""from playwright.sync_api import sync_playwright
import json
import time
import re

headless = {NOTEBOOK_HEADLESS!r}
search_url = {search_url!r}
max_products = {MAX_PRODUCTS}

products = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=headless)
    page = browser.new_page()
    
    print(f"접속 중: {{search_url}}")
    try:
        page.goto(search_url, wait_until="networkidle", timeout=30000)
        time.sleep(2)
        
        # 스크롤하여 동적 콘텐츠 로드 (더 많은 상품 로드)
        for scroll_idx in range(5):
            page.evaluate("window.scrollBy(0, window.innerHeight)")
            time.sleep(0.8)
        
        # 페이지 끝까지 스크롤
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(1)
        
        # 상품 리스트 찾기
        product_list = page.query_selector("ul.search_product_list")
        if not product_list:
            product_list = page.query_selector("#div_product_list")
        
        if product_list:
            # 각 상품 li 요소 찾기
            product_items = product_list.query_selector_all("li")
            print(f"\\n발견된 상품 수: {{len(product_items)}}개")
            
            for item_elem in product_items[:max_products]:  # 최대 개수만큼만 처리
                try:
                    # data 속성에서 정보 추출
                    title = item_elem.get_attribute("data-title") or ""
                    img_url = item_elem.get_attribute("data-img-url") or ""
                    
                    # 상품 링크 찾기
                    product_link = ""
                    link_elem = item_elem.query_selector("a")
                    if link_elem:
                        href = link_elem.get_attribute("href") or ""
                        if href:
                            if href.startswith("http"):
                                product_link = href
                            else:
                                product_link = "https://ssadagu.kr" + href
                    
                    # 판매개수 찾기
                    sales_count = ""
                    try:
                        item_text = item_elem.inner_text()
                        # "판매개수" 뒤의 숫자 찾기
                        sales_match = re.search(r'판매개수\\s*(\\d+)', item_text)
                        if sales_match:
                            sales_count = sales_match.group(1)
                        else:
                            # "개" 앞의 숫자 찾기
                            count_match = re.search(r'(\\d+)\\s*개', item_text)
                            if count_match:
                                sales_count = count_match.group(1)
                    except Exception as e:
                        pass
                    
                    # 가격 정보 추출 (화면에 표시된 가격)
                    displayed_price = ""
                    try:
                        price_selectors = [
                            ".product_price",
                            "[class*='price']",
                            ".price",
                            "div.product_info"
                        ]
                        for selector in price_selectors:
                            price_elem = item_elem.query_selector(selector)
                            if price_elem:
                                price_text = price_elem.inner_text().strip()
                                # 숫자와 "원" 포함된 텍스트 추출
                                price_match = re.search(r'([\\d,]+)\\s*원', price_text)
                                if price_match:
                                    displayed_price = price_match.group(1).replace(",", "")
                                    break
                    except:
                        pass
                    
                    # 최소한 제목이 있어야 유효한 상품
                    if title:
                        product_data = {{
                            "title": title,
                            "price": displayed_price,
                            "product_link": product_link,
                            "thumbnail_url": img_url,
                            "sales_count": sales_count
                        }}
                        products.append(product_data)
                except Exception as e:
                    print(f"  상품 정보 추출 오류: {{e}}")
                    continue
        else:
            print("⚠ 상품 리스트를 찾을 수 없습니다.")
            
    except Exception as e:
        print(f"⚠ 크롤링 오류: {{e}}")
    
    browser.close()
    
    # 결과 저장
    result = {{
        "search_keyword": {SEARCH_KEYWORD!r},
        "search_url": search_url,
        "total_products": len(products),
        "products": products
    }}
    
    # JSON 파일로 저장
    output_json = "ssadagu_search_results.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"\\n✅ 총 {{len(products)}}개 상품 정보 수집 완료")
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    
    # 결과 미리보기
    print(f"\\n📋 수집된 상품 미리보기 (처음 5개):")
    for idx, product in enumerate(products[:5], 1):
        print(f"\\n  {{idx}}. {{product['title'][:50]}}...")
        print(f"     가격: {{product['price']}}원" if product['price'] else "     가격: 정보 없음")
        print(f"     판매개수: {{product['sales_count']}}개" if product['sales_count'] else "     판매개수: 정보 없음")
        print(f"     링크: {{product['product_link'][:60]}}..." if product['product_link'] else "     링크: 정보 없음")
"""

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_by_keyword.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

print("▶ 키워드 기반 상품 검색을 시작합니다...\n")

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass


🔍 검색 키워드: 스마트폰
🔗 검색 URL: https://ssadagu.kr/shop/search.php?ss_tx=%EC%8A%A4%EB%A7%88%ED%8A%B8%ED%8F%B0
📦 최대 수집 개수: 30개

▶ 키워드 기반 상품 검색을 시작합니다...

접속 중: https://ssadagu.kr/shop/search.php?ss_tx=%EC%8A%A4%EB%A7%88%ED%8A%B8%ED%8F%B0

발견된 상품 수: 60개

✅ 총 30개 상품 정보 수집 완료
✅ JSON 파일 저장 완료: ssadagu_search_results.json

📋 수집된 상품 미리보기 (처음 5개):

  1. 2025 새로운 국경 스마트 폰 I16PROMax 안드로이드 전화 AliExpress 핫 ...
     가격: 41800원
     판매개수: 0개
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=90187...

  2. 안드로이드 5G 스마트 폰 i15 ProMax 게임 Netcom 듀얼 카드 P70ProMa...
     가격: 53625원
     판매개수: 0개
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=84744...

  3. 국경 간 Suo Ye XS18Pro 미니 스마트 폰 3.0 "WiFi 블루투스 HD 스크린...
     가격: 34584원
     판매개수: 0개
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=90304...

  4. 새로운 정품 13promax 듀얼 카드 5G 대형 스크린 스마트 폰 16GB 512GB 메...
     가격: 70664원
     판매개수: 0개
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=91377...

  5. 17ProMa

In [ ]:
# Cell 3: 테스트용 - 처음 3개 카테고리만 크롤링
# 전체 크롤링 전에 테스트하기 위한 셀입니다.
import sys
import tempfile
import subprocess
import os
import json
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# categories.json 파일 읽기
categories_file = "categories.json"
if os.path.exists(categories_file):
    with open(categories_file, "r", encoding="utf-8") as f:
        categories_data = json.load(f)
    
    # 모든 카테고리 URL 생성
    category_urls = []
    for category in categories_data.get("categories", []):
        main_cat = category["name"]
        for subcat in category.get("subcategories", []):
            subcat_name = subcat["name"]
            for item in subcat.get("items", []):
                # 검색 URL 생성: ss_tx 파라미터에 검색어 전달
                search_query = f"{subcat_name} {item}"
                url = f"https://ssadagu.kr/shop/search.php?ss_tx={quote(search_query)}"
                category_urls.append({
                    "main_category": main_cat,
                    "sub_category": subcat_name,
                    "item": item,
                    "url": url
                })
    
    # 테스트용: 처음 3개 카테고리만 사용
    category_urls = category_urls[:3]
    
    print(f"테스트용: {len(category_urls)}개 카테고리 URL 생성 완료")
    print(f"예시: {category_urls[0] if category_urls else '없음'}")
else:
    print(f"⚠ {categories_file} 파일을 찾을 수 없습니다.")
    category_urls = []

# 크롤링 스크립트 생성
crawl_script = f"""from playwright.sync_api import sync_playwright
import json
import time
import re
from urllib.parse import quote

headless = {NOTEBOOK_HEADLESS!r}
category_urls_json = {json.dumps(category_urls, ensure_ascii=False)!r}
category_urls = json.loads(category_urls_json)

all_products = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=headless)
    page = browser.new_page()
    
    total_categories = len(category_urls)
    
    for idx, cat_info in enumerate(category_urls, 1):
        category_url = cat_info["url"]
        main_cat = cat_info["main_category"]
        sub_cat = cat_info["sub_category"]
        item = cat_info["item"]
        
        print(f"\\n[{{idx}}/{{total_categories}}] 크롤링 중: {{main_cat}} > {{sub_cat}} > {{item}}")
        print(f"  URL: {{category_url}}")
        
        try:
            page.goto(category_url, wait_until="networkidle", timeout=30000)
            time.sleep(2)
            
            # 스크롤하여 동적 콘텐츠 로드 (더 많은 상품 로드)
            for scroll_idx in range(5):
                page.evaluate("window.scrollBy(0, window.innerHeight)")
                time.sleep(0.8)
            
            # 페이지 끝까지 스크롤
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            time.sleep(1)
            
            # 상품 리스트 찾기
            product_list = page.query_selector("ul.search_product_list")
            if not product_list:
                product_list = page.query_selector("#div_product_list")
            
            if product_list:
                # 각 상품 li 요소 찾기
                product_items = product_list.query_selector_all("li")
                print(f"  발견된 상품 수: {{len(product_items)}}개")
                
                for item_elem in product_items:
                    try:
                        # data 속성에서 정보 추출
                        product_id = item_elem.get_attribute("data-gs-id") or ""
                        title = item_elem.get_attribute("data-title") or ""
                        img_url = item_elem.get_attribute("data-img-url") or ""
                        
                        # 상품 링크 찾기
                        product_link = ""
                        link_elem = item_elem.query_selector("a")
                        if link_elem:
                            href = link_elem.get_attribute("href") or ""
                            if href:
                                if href.startswith("http"):
                                    product_link = href
                                else:
                                    product_link = "https://ssadagu.kr" + href
                        
                        # 판매개수 찾기 - 더 정확한 추출
                        sales_count = ""
                        try:
                            # 전체 텍스트에서 "판매개수" 또는 "개" 패턴 찾기
                            item_text = item_elem.inner_text()
                            # "판매개수" 뒤의 숫자 찾기
                            sales_match = re.search(r'판매개수\\s*(\\d+)', item_text)
                            if sales_match:
                                sales_count = sales_match.group(1)
                            else:
                                # "개" 앞의 숫자 찾기 (판매개수 컨텍스트에서)
                                count_match = re.search(r'(\\d+)\\s*개', item_text)
                                if count_match:
                                    sales_count = count_match.group(1)
                        except Exception as e:
                            pass
                        
                        # 가격 정보 추출 (화면에 표시된 가격)
                        displayed_price = ""
                        try:
                            # 여러 가능한 가격 셀렉터 시도
                            price_selectors = [
                                ".product_price",
                                "[class*='price']",
                                ".price",
                                "div.product_info"
                            ]
                            for selector in price_selectors:
                                price_elem = item_elem.query_selector(selector)
                                if price_elem:
                                    price_text = price_elem.inner_text().strip()
                                    # 숫자와 "원" 포함된 텍스트 추출
                                    price_match = re.search(r'([\\d,]+)\\s*원', price_text)
                                    if price_match:
                                        displayed_price = price_match.group(1).replace(",", "")
                                        break
                        except:
                            pass
                        
                        # 최소한 ID나 제목이 있어야 유효한 상품
                        if product_id or title:
                            product_data = {{
                                "main_category": main_cat,
                                "sub_category": sub_cat,
                                "item": item,
                                "category_url": category_url,
                                "product_id": product_id,
                                "title": title,
                                "product_link": product_link,
                                "thumbnail_url": img_url,
                                "displayed_price": displayed_price,
                                "sales_count": sales_count
                            }}
                            all_products.append(product_data)
                    except Exception as e:
                        print(f"    상품 정보 추출 오류: {{e}}")
                        continue
            else:
                print(f"  ⚠ 상품 리스트를 찾을 수 없습니다.")
                
        except Exception as e:
            print(f"  ⚠ 카테고리 크롤링 오류: {{e}}")
            continue
        
        # 요청 간 딜레이 (서버 부하 방지)
        time.sleep(1)
    
    browser.close()
    
    # 결과 저장
    result = {{
        "total_categories": total_categories,
        "total_products": len(all_products),
        "products": all_products
    }}
    
    # JSON 파일로 저장
    output_json = "ssadagu_products_test.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"\\n✅ 총 {{len(all_products)}}개 상품 정보 수집 완료")
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    print(f"\\n카테고리별 상품 수:")
    if all_products:
        # 카테고리별 상품 수 집계
        category_counts = {{}}
        for product in all_products:
            key = (product["main_category"], product["sub_category"], product["item"])
            category_counts[key] = category_counts.get(key, 0) + 1
        for (main, sub, item), count in sorted(category_counts.items()):
            print(f"  {{main}} > {{sub}} > {{item}}: {{count}}개")
"""

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_test_categories.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

print("▶ 테스트용: 처음 3개 카테고리만 크롤링을 시작합니다...")
print("⚠ 이 셀은 테스트용입니다. 전체 크롤링은 Cell 4를 사용하세요.\n")

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass


테스트용: 3개 카테고리 URL 생성 완료
예시: {'main_category': '패션의류/이너웨어', 'sub_category': '남성의류', 'item': '셔츠', 'url': 'https://ssadagu.kr/shop/search.php?ss_tx=%EB%82%A8%EC%84%B1%EC%9D%98%EB%A5%98%20%EC%85%94%EC%B8%A0'}
▶ 테스트용: 처음 3개 카테고리만 크롤링을 시작합니다...
⚠ 이 셀은 테스트용입니다. 전체 크롤링은 Cell 4를 사용하세요.


[1/3] 크롤링 중: 패션의류/이너웨어 > 남성의류 > 셔츠
  URL: https://ssadagu.kr/shop/search.php?ss_tx=%EB%82%A8%EC%84%B1%EC%9D%98%EB%A5%98%20%EC%85%94%EC%B8%A0
  발견된 상품 수: 60개

[2/3] 크롤링 중: 패션의류/이너웨어 > 남성의류 > 티셔츠
  URL: https://ssadagu.kr/shop/search.php?ss_tx=%EB%82%A8%EC%84%B1%EC%9D%98%EB%A5%98%20%ED%8B%B0%EC%85%94%EC%B8%A0
  발견된 상품 수: 60개

[3/3] 크롤링 중: 패션의류/이너웨어 > 남성의류 > 후드/맨투맨
  URL: https://ssadagu.kr/shop/search.php?ss_tx=%EB%82%A8%EC%84%B1%EC%9D%98%EB%A5%98%20%ED%9B%84%EB%93%9C/%EB%A7%A8%ED%88%AC%EB%A7%A8
  발견된 상품 수: 60개

✅ 총 180개 상품 정보 수집 완료
✅ JSON 파일 저장 완료: ssadagu_products_test.json

카테고리별 상품 수:
  패션의류/이너웨어 > 남성의류 > 셔츠: 60개
  패션의류/이너웨어 > 남성의류 > 티셔츠: 60개
  패션의류/이너웨어 > 남성의류 > 후드/맨투맨: 60개



In [ ]:
# Cell 4: categories.json을 활용한 전체 카테고리 자동 크롤링
# 모든 카테고리의 상품 정보를 자동으로 수집하고 JSON으로 저장합니다.
import sys
import tempfile
import subprocess
import os
import json
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# categories.json 파일 읽기
categories_file = "categories.json"
if os.path.exists(categories_file):
    with open(categories_file, "r", encoding="utf-8") as f:
        categories_data = json.load(f)
    
    # 모든 카테고리 URL 생성
    category_urls = []
    for category in categories_data.get("categories", []):
        main_cat = category["name"]
        for subcat in category.get("subcategories", []):
            subcat_name = subcat["name"]
            for item in subcat.get("items", []):
                # 검색 URL 생성: ss_tx 파라미터에 검색어 전달
                search_query = f"{subcat_name} {item}"
                url = f"https://ssadagu.kr/shop/search.php?ss_tx={quote(search_query)}"
                category_urls.append({
                    "main_category": main_cat,
                    "sub_category": subcat_name,
                    "item": item,
                    "url": url
                })
    
    print(f"총 {len(category_urls)}개 카테고리 URL 생성 완료")
    print(f"예시: {category_urls[0] if category_urls else '없음'}")
else:
    print(f"⚠ {categories_file} 파일을 찾을 수 없습니다.")
    category_urls = []

# 크롤링 스크립트 생성
crawl_script = f"""from playwright.sync_api import sync_playwright
import json
import time
import re
from urllib.parse import quote

headless = {NOTEBOOK_HEADLESS!r}
category_urls_json = {json.dumps(category_urls, ensure_ascii=False)!r}
category_urls = json.loads(category_urls_json)

all_products = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=headless)
    page = browser.new_page()
    
    total_categories = len(category_urls)
    
    for idx, cat_info in enumerate(category_urls, 1):
        category_url = cat_info["url"]
        main_cat = cat_info["main_category"]
        sub_cat = cat_info["sub_category"]
        item = cat_info["item"]
        
        print(f"\\n[{{idx}}/{{total_categories}}] 크롤링 중: {{main_cat}} > {{sub_cat}} > {{item}}")
        print(f"  URL: {{category_url}}")
        
        try:
            page.goto(category_url, wait_until="networkidle", timeout=30000)
            time.sleep(2)
            
            # 스크롤하여 동적 콘텐츠 로드 (더 많은 상품 로드)
            for scroll_idx in range(5):
                page.evaluate("window.scrollBy(0, window.innerHeight)")
                time.sleep(0.8)
            
            # 페이지 끝까지 스크롤
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            time.sleep(1)
            
            # 상품 리스트 찾기
            product_list = page.query_selector("ul.search_product_list")
            if not product_list:
                product_list = page.query_selector("#div_product_list")
            
            if product_list:
                # 각 상품 li 요소 찾기
                product_items = product_list.query_selector_all("li")
                print(f"  발견된 상품 수: {{len(product_items)}}개")
                
                for item_elem in product_items:
                    try:
                        # data 속성에서 정보 추출
                        product_id = item_elem.get_attribute("data-gs-id") or ""
                        title = item_elem.get_attribute("data-title") or ""
                        img_url = item_elem.get_attribute("data-img-url") or ""
                        
                        # 상품 링크 찾기
                        product_link = ""
                        link_elem = item_elem.query_selector("a")
                        if link_elem:
                            href = link_elem.get_attribute("href") or ""
                            if href:
                                if href.startswith("http"):
                                    product_link = href
                                else:
                                    product_link = "https://ssadagu.kr" + href
                        
                        # 판매개수 찾기 - 더 정확한 추출
                        sales_count = ""
                        try:
                            # 전체 텍스트에서 "판매개수" 또는 "개" 패턴 찾기
                            item_text = item_elem.inner_text()
                            # "판매개수" 뒤의 숫자 찾기
                            sales_match = re.search(r'판매개수\\s*(\\d+)', item_text)
                            if sales_match:
                                sales_count = sales_match.group(1)
                            else:
                                # "개" 앞의 숫자 찾기 (판매개수 컨텍스트에서)
                                count_match = re.search(r'(\\d+)\\s*개', item_text)
                                if count_match:
                                    sales_count = count_match.group(1)
                        except Exception as e:
                            pass
                        
                        # 가격 정보 추출 (화면에 표시된 가격)
                        displayed_price = ""
                        try:
                            # 여러 가능한 가격 셀렉터 시도
                            price_selectors = [
                                ".product_price",
                                "[class*='price']",
                                ".price",
                                "div.product_info"
                            ]
                            for selector in price_selectors:
                                price_elem = item_elem.query_selector(selector)
                                if price_elem:
                                    price_text = price_elem.inner_text().strip()
                                    # 숫자와 "원" 포함된 텍스트 추출
                                    price_match = re.search(r'([\\d,]+)\\s*원', price_text)
                                    if price_match:
                                        displayed_price = price_match.group(1).replace(",", "")
                                        break
                        except:
                            pass
                        
                        # 최소한 ID나 제목이 있어야 유효한 상품
                        if product_id or title:
                            product_data = {{
                                "main_category": main_cat,
                                "sub_category": sub_cat,
                                "item": item,
                                "category_url": category_url,
                                "product_id": product_id,
                                "title": title,
                                "product_link": product_link,
                                "thumbnail_url": img_url,
                                "displayed_price": displayed_price,
                                "sales_count": sales_count
                            }}
                            all_products.append(product_data)
                    except Exception as e:
                        print(f"    상품 정보 추출 오류: {{e}}")
                        continue
            else:
                print(f"  ⚠ 상품 리스트를 찾을 수 없습니다.")
                
        except Exception as e:
            print(f"  ⚠ 카테고리 크롤링 오류: {{e}}")
            continue
        
        # 요청 간 딜레이 (서버 부하 방지)
        time.sleep(1)
    
    browser.close()
    
    # 결과 저장
    result = {{
        "total_categories": total_categories,
        "total_products": len(all_products),
        "products": all_products
    }}
    
    # JSON 파일로 저장
    output_json = "ssadagu_products.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"\\n✅ 총 {{len(all_products)}}개 상품 정보 수집 완료")
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    print(f"\\n카테고리별 상품 수:")
    if all_products:
        # 카테고리별 상품 수 집계
        category_counts = {{}}
        for product in all_products:
            key = (product["main_category"], product["sub_category"], product["item"])
            category_counts[key] = category_counts.get(key, 0) + 1
        for (main, sub, item), count in sorted(category_counts.items()):
            print(f"  {{main}} > {{sub}} > {{item}}: {{count}}개")
"""

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_all_categories.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

print("▶ 전체 카테고리 상품 크롤링을 시작합니다...")
print("⚠ 주의: 모든 카테고리를 크롤링하므로 시간이 오래 걸릴 수 있습니다.")
print("⚠ 중단하려면 커널을 중지하세요.\n")

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass
